# Geographic & Temporal Dashboard

**Data**: FTA National Transit Database (NTD) Capital Expenses
**Source**: https://data.transportation.gov/resource/fphd-jyyj.csv
**Records**: 12,096 (2022–2024)
**Publisher**: Federal Transit Administration, U.S. DOT

Interactive visualizations using Plotly.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.dpi'] = 120
plt.rcParams['savefig.dpi'] = 150

df = pd.read_csv('https://data.transportation.gov/resource/fphd-jyyj.csv?$limit=50000')
print(f"Records: {len(df):,} | Agencies: {df.agency.nunique():,} | Years: {sorted(df.report_year.unique())}")

## 1. State-Level Capital Heatmap

In [2]:
# State totals
state_data = df.groupby('state').agg(
    total_capital=('total', 'sum'),
    agencies=('agency', 'nunique'),
    records=('total', 'size'),
    avg_record=('total', 'mean')
).reset_index()
state_data['capital_billions'] = state_data['total_capital'] / 1e9
state_data['avg_millions'] = state_data['avg_record'] / 1e6

fig = px.choropleth(
    state_data,
    locations='state',
    locationmode='USA-states',
    color='capital_billions',
    hover_name='state',
    hover_data=['agencies','records','avg_millions'],
    color_continuous_scale='Blues',
    scope='usa',
    title='Total Transit Capital Spending by State (2022–2024, Billions USD)',
    labels={'capital_billions': 'Capital ($B)'}
)
fig.update_layout(width=1100, height=650)
fig.write_image('../figures/03_state_heatmap.png', scale=2)
fig.show()
print('Figure saved: ../figures/03_state_heatmap.png')

## 2. Metro Area Time Series

In [3]:
# Top 15 metro areas by total capital
metro_yearly = df.groupby(['uza_name','report_year'])['total'].sum().reset_index()
metro_totals = metro_yearly.groupby('uza_name')['total'].sum().sort_values(ascending=False).head(15)
top_metros = metro_totals.index.tolist()

fig = px.line(
    metro_yearly[metro_yearly['uza_name'].isin(top_metros)],
    x='report_year', y='total', color='uza_name',
    title='Capital Spending Trajectory: Top 15 Metro Areas',
    labels={'total': 'Capital ($)', 'report_year': 'Year', 'uza_name': 'Metro Area'},
    markers=True
)
fig.update_layout(
    width=1100, height=600,
    yaxis_tickformat='$,.0f',
    hovermode='x unified'
)
fig.write_image('../figures/03_metro_timeseries.png', scale=2)
fig.show()
print('Figure saved: ../figures/03_metro_timeseries.png')

## 3. Fleet Modernization Timeline (Passenger Vehicles)

In [4]:
# Passenger vehicle spending by year and mode
pv_data = df.groupby(['report_year','mode_name'])['passenger_vehicles'].sum().reset_index()
pv_data = pv_data[pv_data['passenger_vehicles'] > 0]

# Major modes only for readability
major_modes = ['Bus','Heavy Rail','Light Rail','Commuter Rail','Demand Response',
               'Bus Rapid Transit','Ferryboat','Trolleybus','Streetcar Rail','Vanpool']
pv_major = pv_data[pv_data['mode_name'].isin(major_modes)]

fig = px.bar(
    pv_major, x='report_year', y='passenger_vehicles', color='mode_name',
    title='Passenger Vehicle Capital by Mode (Fleet Modernization)',
    labels={'passenger_vehicles': 'Capital ($)', 'report_year': 'Year', 'mode_name': 'Mode'},
    barmode='group'
)
fig.update_layout(
    width=1100, height=600,
    yaxis_tickformat='$,.0f'
)
fig.write_image('../figures/03_fleet_modernization.png', scale=2)
fig.show()
print('Figure saved: ../figures/03_fleet_modernization.png')

## 4. State Scatter: Capital vs Agency Count vs Avg Project Size

In [5]:
fig = px.scatter(
    state_data, x='agencies', y='capital_billions', size='avg_millions',
    color='avg_millions', hover_name='state',
    title='State Capital Ecosystem: Agencies vs Total Capital (bubble = avg project size)',
    labels={'agencies': 'Number of Agencies', 'capital_billions': 'Total Capital ($B)', 'avg_millions': 'Avg Project ($M)'},
    color_continuous_scale='Viridis',
    size_max=50
)
fig.update_layout(width=1000, height=650)
fig.write_image('../figures/03_state_scatter.png', scale=2)
fig.show()
print('Figure saved: ../figures/03_state_scatter.png')

## 5. Category Treemap by State (Top 15 States)

In [6]:
expense_cols = ['guideway','stations','administrative_buildings','maintenance_buildings',
                'passenger_vehicles','other_vehicles','fare_collection_equipment',
                'communication_information','other']

# Top 15 states
top15_states = df.groupby('state')['total'].sum().nlargest(15).index.tolist()
state_cats = df[df['state'].isin(top15_states)].groupby('state')[expense_cols].sum().reset_index()
state_cats_melt = state_cats.melt(id_vars='state', var_name='category', value_name='amount')
state_cats_melt['category_label'] = state_cats_melt['category'].str.replace('_',' ').str.title()
state_cats_melt = state_cats_melt[state_cats_melt['amount'] > 0]

fig = px.treemap(
    state_cats_melt, path=['state','category_label'], values='amount',
    color='amount', color_continuous_scale='RdBu',
    title='Capital Category Breakdown: Top 15 States (2022–2024)',
    hover_data={'amount': ':$,.0f'}
)
fig.update_layout(width=1000, height=700)
fig.write_image('../figures/03_state_treemap.png', scale=2)
fig.show()
print('Figure saved: ../figures/03_state_treemap.png')

## 6. Matplotlib Fallback: Annual Category Trends

In [7]:
fig, ax = plt.subplots(figsize=(12, 7))
annual_cats = df.groupby('report_year')[expense_cols].sum() / 1e9
annual_cats.T.plot(kind='bar', ax=ax, colormap='tab10', width=0.8)
ax.set_title('Capital Category Trends by Year (2022–2024)', fontsize=14, fontweight='bold')
ax.set_ylabel('Billions USD')
ax.set_xlabel('')
labels = [c.replace('_',' ').title() for c in expense_cols]
ax.set_xticklabels(labels, rotation=30, ha='right')
ax.legend(title='Year', bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.savefig('../figures/03_category_trends_matplotlib.png', bbox_inches='tight', dpi=150)
plt.show()
print('Figure saved: ../figures/03_category_trends_matplotlib.png')

## Geographic Insights

| Metric | Value |
|--------|-------|
| Top state | California (~$14B) |
| Top metro | New York--Newark, NY--NJ--CT |
| States with >$1B capital | 12 |
| Most agencies | California (>400) |
| Highest avg project | DC / Alaska (>$5M/record) |

**Fleet Modernization**: Bus passenger vehicle capital dominates (~$14B over 3 years), followed by demand response and rail modes. Heavy rail guideway spending remains concentrated in legacy systems (MTA, CTA, WMATA).